In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("spam.csv")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df.groupby('Category').describe()

Message                                                            \
           count unique                                                top   
Category                                                                     
ham         4825   4516                             Sorry, I'll call later   
spam         747    641  Please call our customer service representativ...   

               
         freq  
Category       
ham        30  
spam        4

In [4]:
df['spam']=df['Category'].apply(lambda x: 1 if x=='spam' else 0)
df.head()

,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.Message,df.spam, test_size=0.2)

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
v = CountVectorizer()
X_train_count = v.fit_transform(X_train.values)
X_train_count.toarray()[:2]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [7]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train_count,y_train)

MultinomialNB()

In [8]:
emails = [
    'Hey mohan, can we get together to watch footbal game tomorrow?',
    'Upto 20% discount on parking, exclusive offer just for you. Dont miss this reward!'
]
emails_count = v.transform(emails)
model.predict(emails_count)

array([0, 1])

In [9]:
X_test_count = v.transform(X_test)
model.score(X_test_count, y_test)

0.9910313901345291

**Sklearn Pipeline**

In [10]:
from sklearn.pipeline import Pipeline
clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

In [ ]:
clf.fit(X_train, y_train)

Pipeline(memory=None,
     steps=[('vectorizer', CountVectorizer(analyzer='word', binary=False, decode_error='strict',
        dtype=<class 'numpy.int64'>, encoding='utf-8', input='content',
        lowercase=True, max_df=1.0, max_features=None, min_df=1,
        ngram_range=(1, 1), preprocessor=None, stop_words=None,
        strip_accents=None, token_pattern='(?u)\\b\\w\\w+\\b',
        tokenizer=None, vocabulary=None)), ('nb', MultinomialNB(alpha=1.0, class_prior=None, fit_prior=True))])

In [ ]:
clf.score(X_test,y_test)

0.9827709978463748

In [ ]:
clf.predict(emails)

array([0, 1], dtype=int64)

### Machine Learning Tutorial - Naive Bayes: Exercise

Use wine dataset from sklearn.datasets to classify wines into 3 categories. Load the dataset and split it into test and train. After that train the model using Gaussian and Multinominal classifier and post which model performs better. Use the trained model to perform some predictions on test data.

In [11]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB

In [12]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

In [13]:
print("Target categories (Wine classes):", wine.target_names)
print(df.head())

Target categories (Wine classes): ['class_0' 'class_1' 'class_2']
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04 

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('target', axis='columns'),
    df.target,
    test_size=0.2,
    random_state=42
)

In [15]:
gnb_model = GaussianNB()
gnb_model.fit(X_train, y_train)
gnb_score = gnb_model.score(X_test, y_test)

In [16]:
mnb_model = MultinomialNB()
mnb_model.fit(X_train, y_train)
mnb_score = mnb_model.score(X_test, y_test)

In [17]:
print("\n--- Model Performance Comparison ---")
print(f"Gaussian Naive Bayes Accuracy:    {gnb_score:.4f}")
print(f"Multinomial Naive Bayes Accuracy: {mnb_score:.4f}")


--- Model Performance Comparison ---
Gaussian Naive Bayes Accuracy:    1.0000
Multinomial Naive Bayes Accuracy: 0.8889


In [18]:
best_model = gnb_model if gnb_score > mnb_score else mnb_model
predictions = best_model.predict(X_test)

print("\n--- Sample Predictions (Using Best Model) ---")
comparison_df = pd.DataFrame({
    'Actual Class': y_test.values,
    'Predicted Class': predictions
})
print(comparison_df.head(10))


--- Sample Predictions (Using Best Model) ---
   Actual Class  Predicted Class
0             0                0
1             0                0
2             2                2
3             0                0
4             1                1
5             0                0
6             1                1
7             2                2
8             1                1
9             2                2
